Note: Set your HF_TOKEN

In [ ]:
# Setup and Imports
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

import os
import json
import pandas as pd
import torch
from datasets import Dataset
from google.colab import drive, userdata
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)


print(" Logging into Hugging Face...")
from huggingface_hub import login
try:
    login(token=userdata.get('HF_TOKEN'))
    print(" Successfully logged in.")
except Exception as e:
    print(f"Login failed. Please ensure your HF_TOKEN secret is set correctly.")


print(" Mounting Google Drive...")
drive.mount('/content/drive')
GDRIVE_PATH = '/content/drive/MyDrive/'
DATA_DIR = os.path.join(GDRIVE_PATH, 'FinalData/')
MODELS_DIR = os.path.join(GDRIVE_PATH, 'LargeModelsFinal/Personality_Brain/')
CHECKPOINTS_DIR = os.path.join(GDRIVE_PATH, 'HF_Checkpoints', 'Gemma_Personality_Brain')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Setup complete. Using device: {device.upper()}")


In [ ]:
# Prepare Dataset
print("\n Loading and preparing Pandora dataset...")
df_pandora = pd.read_csv(os.path.join(DATA_DIR, 'pandora_clean.csv')).dropna()

def score_to_level(score):
    if score > 66: return "High"
    if score < 34: return "Low"
    return "Medium"

def create_instructional_text(row):
    personality_profile = (
        f"Openness: {score_to_level(row['openness'])}, "
        f"Conscientiousness: {score_to_level(row['conscientiousness'])}, "
        f"Extraversion: {score_to_level(row['extraversion'])}, "
        f"Agreeableness: {score_to_level(row['agreeableness'])}, "
        f"Neuroticism: {score_to_level(row['neuroticism'])}"
    )
    prompt = f"You are a chatbot. Your personality is: {personality_profile}. A user says: 'Tell me about yourself.' Respond as yourself."
    response = row['text']
    return f"<s>[INST] {prompt} [/INST] {response} </s>"

df_pandora['text'] = df_pandora.apply(create_instructional_text, axis=1)
pandora_dataset_hf = Dataset.from_pandas(df_pandora[['text']])


model_name = "google/gemma-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

print("Tokenizing data efficiently...")
tokenized_dataset = pandora_dataset_hf.map(tokenize_function, batched=True, remove_columns=["text"])
print(" Dataset prepared and tokenized.")


In [ ]:
# Loading Model and LoRA
print("\n Loading base Gemma model with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
print(" Base model loaded and LoRA configured.")


In [ ]:
# Fine-Tuning
print("\n Preparing for fine-tuning...")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=CHECKPOINTS_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_steps=5000,
    optim="paged_adamw_8bit",
    fp16=True,
    logging_steps=100,
    save_steps=500,
    report_to="none",
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator,
)

print("   -> Starting fine-tuning run for 5000 steps...")
trainer.train()
print(" Fine-tuning complete.")
trainer.save_model(MODELS_DIR)
print(f"Final 'Personality Brain' model saved to: {MODELS_DIR}")

In [ ]:
# Testing
print("\n Testing the final fine-tuned model...")

base_model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = PeftModel.from_pretrained(base_model, MODELS_DIR)
print(" Model ready.")


test_personality_profile = (
    "Openness: High, Conscientiousness: Low, Extraversion: High, "
    "Agreeableness: Low, Neuroticism: High"
)
prompt = f"You are a chatbot. Your personality is: {test_personality_profile}. A user says: 'What's your plan for the weekend?' Respond as yourself."
input_text = f"<s>[INST] {prompt} [/INST]"


print("\nGenerating response...")
inputs = tokenizer(input_text, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
cleaned_response = response.split("[/INST]")[-1].strip().replace("</s>", "").strip()

print("\n--- Model's Response ---")
print(cleaned_response)